## Homework 04: Linear Regression and Model Development

#### Due Date: Sunday, September 27th at 11:59pm (with 2 hour plus 1 minute grace period)

#### The late policy is described in the DX 603 Syllabus.

### Introduction

In Homework 03, you used exploratory data analysis to identify characteristics of the **Ames Housing dataset** that might matter for modeling. You found skewed numerical features, examined alternative representations of a high-cardinality categorical feature, and identified preprocessing choices that seemed reasonable.

In this homework, you will continue with the Ames Dataset and test several of those ideas by building linear regression models. The central question is:

> **Does a proposed feature-engineering change improve validation performance enough that we should retain it?**

You will first construct a baseline regression model. You will then test a log transformation, compare two encodings of `Neighborhood`, and test a quadratic relationship. Finally, you will select a model using the validation set, refit it on the complete development set, and evaluate it once on the untouched test set.

The primary model-selection metric in this homework is **mean absolute error (MAE)**, measured in dollars.

> You are **not expected to write every scikit-learn pipeline from scratch**. Reuse and adapt code from the Week 4 Coding Notebook, and use AI to help you generate or modify code when appropriate. You are responsible for running the code, checking that it follows the instructions, and interpreting the results.

### Grading

- All homeworks in DX 603 are worth **80 points**.
- Problems 1--5 are worth **12 points each**.
- Problems 6 and 7 are worth **10 points each**.
- Multi-part problems will have approximately equal points for each subpart.

You may submit a regrade request on Gradescope after your grades are released. The instructor will review the request and make any appropriate adjustment.

### Pipeline Requirement

All regression models in this homework must be built using a scikit-learn `Pipeline` or `make_pipeline`. When different groups of features require different preprocessing steps, use a `ColumnTransformer`. These have been covered in this week's coding video and notebook; you are free to use any code developed there. 

All preprocessing must be learned from the fitting data as part of the pipeline. Do **not** preprocess the complete dataset before splitting it.

- Use the **training set** to fit candidate models.
- Use the **validation set** to compare models and make modeling decisions.
- Leave the **test set untouched until Problem 5**.

**Important:** Because the imputer, scaler, and encoders are inside the pipeline, fitting a candidate model on X_train means that all preprocessing parameters—such as medians, most-frequent categories, scaling values, and category frequencies—are learned from the training set only. The validation and test sets are transformed using those learned values; they are not used to fit the preprocessing steps.

In [1]:
# Useful imports and utilities

import os
import warnings
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
    StandardScaler,
)


warnings.filterwarnings("ignore")

random_seed = 42

In [2]:
# ==================================================
# Download and load the Ames Housing data
# ==================================================

filename = "AmesHousing.txt"
url = "https://jse.amstat.org/v19n3/decock/AmesHousing.txt"

if not os.path.exists(filename):
    print(f"Downloading {filename}...")
    response = requests.get(url)
    response.raise_for_status()

    with open(filename, "wb") as f:
        f.write(response.content)

    print(f"{filename} downloaded.")
else:
    print(f"{filename} already exists. Skipping download.")

df = pd.read_csv(filename, sep="\t")

display(df.head())
print(f"Dataset shape: {df.shape}")

AmesHousing.txt already exists. Skipping download.


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


Dataset shape: (2930, 82)


## Problem 1: Build a Baseline Linear Regression Pipeline

In this problem, you will build a baseline linear regression model using all of the available predictors in the Ames Housing dataset. This model will provide a baseline against which the models developed in later problems can be compared.

First, create an approximately **80/10/10 train/validation/test split**. Then construct preprocessing pipelines for the numerical and categorical predictors and fit a linear regression model.

### Part A Graded Answer: Create the Data Splits

Separate the complete dataset into predictors `X_all` and target `y_all`, where the target is `SalePrice`.
Drop the identifier columns `Order` and `PID` when you create `X_all`. 

Then create an approximately **80/10/10 train/validation/test split**:

1. Reserve **10% of the complete dataset** as the test set.
2. From the remaining 90% development data, reserve **1/9 for validation**.
3. Use `random_state=random_seed` for both splits.

Create:

* `X_all`, `y_all`
* `X_train`, `y_train`
* `X_development`, `y_development` 
* `X_validation`, `y_validation`
* `X_test`, `y_test`

Assign:

```python
a1a = 0    # Replace with an expression that returns the number of observations in the training set
```

In [3]:
# Your code here, as more cells as needed



In [4]:
# Your answer here, NOT in the next cell

a1a = 0              # Replace with an expression that returns the number of observations in the training set


In [5]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a1a = {a1a}')

a1a = 0


### Part B Graded Answer: Build the Preprocessor

The  **Ames Housing Data Documentation** (downloaded as `DataDescription.txt` in HW03) classifies the predictors as **continuous, discrete, nominal, or ordinal**. 

For modeling, we will treat the predictors as three groups:

* **Numerical predictors:** continuous and discrete variables, together with Overall Qual and Overall Cond, which are ordinal ratings already represented numerically;
* **Nominal categorical predictors:** categorical variables with no natural ordering;
* **Ordinal categorical predictors:** the remaining categorical variables whose values have a meaningful ordering.

The lists of predictors and the category orderings for the ordinal predictors are provided in the code below.

1. Create a preprocessing pipeline for the **numerical predictors** that:

   * replaces missing values with the **median**;
   * standardizes the predictors using `StandardScaler`.

2. Create a preprocessing pipeline for the **nominal categorical predictors** that:

   * replaces missing values with the **most frequent** category;
   * one-hot encodes the predictors using `OneHotEncoder(handle_unknown="ignore")`.

3. Create a preprocessing pipeline for the **ordinal categorical predictors** that:

   * handles missing values as specified in the provided category definitions;
   * uses `OrdinalEncoder` to represent the categories according to their natural ordering.

4. Combine the numerical, nominal, and ordinal pipelines using a `ColumnTransformer` named `baseline_preprocessor`.

5. Fit `baseline_preprocessor` to `X_train` and transform the training data, storing the result as `X_train_preprocessed`.

Print the number of original predictors and the number of features after preprocessing.

Assign:

```python
a1b = 0    # Replace with an expression that returns the number of features after preprocessing
```

**Note:** The coding notebook and video contain template code that you can use to solve this problem.



In [6]:
# Feature groups from the Ames Housing Data Documentation

# Numerical predictors: continuous and discrete variables
numerical_features = [
    "Lot Frontage", "Lot Area", "Overall Qual", "Overall Cond", "Year Built", "Year Remod/Add",
    "Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF", "1st Flr SF",
    "2nd Flr SF", "Low Qual Fin SF", "Gr Liv Area", "Bsmt Full Bath", "Bsmt Half Bath", "Full Bath",
    "Half Bath", "Bedroom AbvGr", "Kitchen AbvGr", "TotRms AbvGrd", "Fireplaces", "Garage Yr Blt",
    "Garage Cars", "Garage Area", "Wood Deck SF", "Open Porch SF", "Enclosed Porch", "3Ssn Porch",
    "Screen Porch", "Pool Area", "Misc Val", "Mo Sold", "Yr Sold"
]

# Nominal categorical predictors
nominal_features = [
    "MS SubClass", "MS Zoning", "Street", "Alley", "Land Contour", "Lot Config", "Neighborhood",
    "Condition 1", "Condition 2", "Bldg Type", "House Style", "Roof Style", "Roof Matl", "Exterior 1st",
    "Exterior 2nd", "Mas Vnr Type", "Foundation", "Heating", "Central Air", "Garage Type",
    "Misc Feature", "Sale Type", "Sale Condition"
]

# Ordinal categorical predictors
ordinal_features = [
    "Lot Shape", "Utilities", "Land Slope", "Exter Qual", "Exter Cond", "Bsmt Qual", "Bsmt Cond",
    "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2", "Heating QC", "Electrical", "Kitchen Qual",
    "Functional", "Fireplace Qu", "Garage Finish", "Garage Qual", "Garage Cond", "Paved Drive",
    "Pool QC", "Fence"
]

# Categories for ordinal predictors, ordered from lowest to highest
ordinal_categories = [
    ["IR3", "IR2", "IR1", "Reg"],                                      # Lot Shape
    ["ELO", "NoSeWa", "NoSewr", "AllPub"],                            # Utilities
    ["Sev", "Mod", "Gtl"],                                             # Land Slope
    ["Po", "Fa", "TA", "Gd", "Ex"],                                   # Exter Qual
    ["Po", "Fa", "TA", "Gd", "Ex"],                                   # Exter Cond
    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],                             # Bsmt Qual
    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],                             # Bsmt Cond
    ["NA", "No", "Mn", "Av", "Gd"],                                   # Bsmt Exposure
    ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],                 # BsmtFin Type 1
    ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],                 # BsmtFin Type 2
    ["Po", "Fa", "TA", "Gd", "Ex"],                                   # Heating QC
    ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],                      # Electrical
    ["Po", "Fa", "TA", "Gd", "Ex"],                                   # Kitchen Qual
    ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],     # Functional
    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],                             # Fireplace Qu
    ["NA", "Unf", "RFn", "Fin"],                                      # Garage Finish
    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],                             # Garage Qual
    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],                             # Garage Cond
    ["N", "P", "Y"],                                                   # Paved Drive
    ["NA", "Fa", "TA", "Gd", "Ex"],                                   # Pool QC
    ["NA", "MnWw", "GdWo", "MnPrv", "GdPrv"]                          # Fence
]

In [7]:
# Your code here, as more cells as needed



In [8]:
# Your answer here, NOT in the next cell

a1b = 0               # Replace with an expression that returns the number of features after preprocessing


In [9]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a1b = {a1b}')

a1b = 0


### Part C Graded Answer: Build and Evaluate the Baseline Model

Create a pipeline named `baseline_model` that combines the `baseline_preprocessor` from Part B with:

```python
LinearRegression()
```

Fit the model using the **training data only**.

Use the fitted model to make predictions for both the **training and validation sets**.

For each set, calculate:

* MAE
* RMSE
* $R^2$

Display the results in a table and briefly compare the training and validation performance.

Assign:

```python
a1c = 0    # Replace with the validation MAE
```

> **Note:** Standardization is not required mathematically for unregularized linear regression. We use it here for numerical stability, consistency with later models, and practice constructing preprocessing pipelines.

In [10]:
# Your code here, add new cells as needed



In [11]:
# Your answer here, NOT in the next cell

a1c = 0              # Replace with an expression that returns the validation MAE


In [12]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a1c = ${a1c:,.2f}')

a1c = $0.00


## Problem 2: Does a Log Transformation Improve the Model?

In Homework 03, you found that `Lot Area` is strongly right-skewed and that a base-10 log transformation makes its distribution substantially more symmetric. We will now test whether that transformation improves prediction.

1. Construct a second pipeline that is identical to the baseline except for its treatment of `Lot Area`:

    * `Lot Area` should pass through median imputation, `FunctionTransformer(np.log10)`, and scaling;
    * the remaining numerical predictors should continue to use median imputation and scaling;
    * the nominal and ordinal categorical predictors should continue to use their baseline preprocessing pipelines.

2. Perform the transformation **inside the pipeline**. Do not modify the original DataFrame.

3. Fit the log-transformed candidate using the **training set only** and evaluate it on the training and validation sets. **Do not use the test set.**

Assign:

```python
a2a = training MAE for the log-transformed model
a2b = validation MAE for the log-transformed model
a2c = representation with the lower validation MAE
```

for `a2c`, use one of these strings exactly

```python
"original"
"log-transformed"
```

In [13]:
# Your code here, as more cells as needed



In [14]:
# Your answer here, NOT in the next cell

a2a = 0              # Replace with an expression that returns the training MAE for the log-transformed model


In [15]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a2a = ${a2a:,.2f}')

a2a = $0.00


In [16]:
# Your answer here, NOT in the next cell

a2b = 0              # Replace with an expression that returns the validation MAE for the log-transformed model


In [17]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a2b = ${a2b:,.2f}')

a2b = $0.00


In [18]:
# Your answer here, NOT in the next cell

a2c = ""              # Replace with appropriate string listed in problem for the lower validation MAE option


In [19]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a2c = {a2c}')

a2c = 


## Problem 3: Does Frequency Encoding Improve the Model?

Neighborhood is a nominal categorical predictor with many different categories. In the baseline model, it is represented using one-hot encoding, which creates a separate binary feature for each neighborhood. For categorical predictors with many possible values, one-hot encoding can substantially increase the number of features. 

An alternative is **frequency encoding.** Instead of creating a separate column for each category, frequency encoding replaces each neighborhood with the proportion of training observations belonging to that neighborhood. This represents Neighborhood using a single numerical feature.

The `FrequencyEncoder` class provided below implements this transformation and can be used inside a scikit-learn pipeline.

In this problem, you will compare these two representations of Neighborhood while keeping the rest of the model unchanged.

In [20]:
# Provided code -- do not change

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Replace each category with its proportion in the fitting data."""

    def fit(self, X, y=None):
        values = pd.Series(np.asarray(X).ravel())

        self.frequencies_ = values.value_counts(
            normalize=True,
            dropna=False,
        )

        return self

    def transform(self, X):
        values = pd.Series(np.asarray(X).ravel())

        encoded = values.map(
            self.frequencies_
        ).fillna(0.0)

        return encoded.to_numpy().reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            name = "feature"
        else:
            name = input_features[0]

        return np.array(
            [f"{name}_frequency"],
            dtype=object,
        )

### Part A Graded Answer: Standard One-Hot Encoding

In the models constructed so far, `Neighborhood` is included among the nominal predictors and is therefore represented using one-hot encoding.

Use the fitted model with the `Lot Area` representation selected in Problem 2.  

Determine how many different Neighborhood categories occur in the training set. Since standard one-hot encoding creates one column for each category, this is the number of columns used to represent `Neighborhood`.

Assign:

    a3a = number of Neighborhood columns produced by standard one-hot encoding

**Hint:** Use `.nunique()`  on the `Neighborhood` feature. 

In [21]:
# Your code here, as more cells as needed



In [22]:
# Your answer here, NOT in the next cell

a3a = 0              # Replace with the number of Neighborhood columns produced by standard one-hot encoding


In [23]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a3a = {a3a}')

a3a = 0


### Parts B & C Graded Answers: Frequency Encoding

Now we'll try an alternative way of representing `Neighborhood` called **frequency encoding**. Instead of creating one binary column for each neighborhood, frequency encoding replaces each neighborhood with the proportion of training observations that belong to that neighborhood. Thus, `Neighborhood` is represented by **one numerical feature** rather than many one-hot features.

Change **only the preprocessing of `Neighborhood`**. All other predictors should use the same preprocessing as in the model selected in Problem 2.

> **Notice:** This kind of targeted change is easy to make when preprocessing and modeling are organized using pipelines.

To construct the new model:

1. Remove `Neighborhood` from the list of nominal predictors processed by the standard one-hot encoder.
2. Give `Neighborhood` its **own transformer** in the `ColumnTransformer`.
3. For `Neighborhood`, perform categorical imputation followed by the provided `FrequencyEncoder`.
4. Leave the preprocessing of all other predictors unchanged.
5. Combine the new preprocessor with `LinearRegression()`.
6. Fit the model using the **training set only** and calculate its validation MAE. **Do not use the test set.**

Compare the validation MAE with the one-hot model selected in Problem 2.

Assign:

```python
a3b = validation MAE for the frequency-encoded model
a3c = representation with the lower validation MAE
```

For `a3c`, use one of these strings exactly:

```python
"one-hot"
"frequency"
```


In [24]:
# Your code here



In [25]:
# Your answer here, NOT in the next cell

a3b = 0    # Replace with the frequency-encoded validation MAE

In [26]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a3b = ${a3b:,.2f}')

a3b = $0.00


In [27]:
# Your answer here, NOT in the next cell

a3c = ""    # Replace with the appropriate string

In [28]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a3c = {a3c}')

a3c = 


## Problem 4: Does a Quadratic Relationship Improve the Model?

The `Overall Qual` feature rates overall material and finish quality on a scale from 1 to 10. Although it is an ordered rating, it is already represented numerically in the Ames dataset. A one-point increase in quality may not correspond to the same change in sale price at every point on the scale, so we will investigate whether adding a **quadratic term** improves the model.

> **Start from the one-hot model selected in Problem 3.** Change **only the treatment of `Overall Qual`**, leaving the preprocessing of all other predictors unchanged.

Construct a `ColumnTransformer` with a separate branch for `Overall Qual`:

* `Overall Qual` $\rightarrow$ median imputation $\rightarrow$ `PolynomialFeatures(degree=2, include_bias=False)` $\rightarrow$ scaling;
* `Lot Area` $\rightarrow$ the representation selected in Problem 2;
* the remaining numerical predictors $\rightarrow$ their existing numerical preprocessing;
* nominal categorical predictors, **including `Neighborhood`**, $\rightarrow$ their existing one-hot preprocessing;
* ordinal categorical predictors $\rightarrow$ their existing ordinal preprocessing.

Only `Overall Qual` should receive the polynomial expansion. Because `include_bias=False`, this branch produces two features:

$$
\text{Overall Qual}, \qquad (\text{Overall Qual})^2.
$$

The final estimator remains `LinearRegression()`.

Fit the quadratic candidate using the **training set only**. Evaluate its performance on both the training and validation sets using MAE, and compare its validation MAE with that of the model selected in Problem 3.

**Do not use the test set.**

### Graded Answers

Assign:

```python
a4a = training MAE for the quadratic model
a4b = validation MAE for the quadratic model
a4c = treatment of Overall Qual with the lower validation MAE
```

For `a4c`, use one of these strings exactly:

```python
"linear"
"quadratic"
```


In [29]:
# Your code here, as more cells as needed



In [30]:
# Your answer here, NOT in the next cell

a4a = 0.0    # Replace with the quadratic-model training MAE

In [31]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a4a = ${a4a:,.2f}')

a4a = $0.00


In [32]:
# Your answer here, NOT in the next cell

a4b = 0.0    # Replace with the quadratic-model validation MAE

In [33]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a4b = ${a4b:,.2f}')

a4b = $0.00


In [34]:
# Your answer here, NOT in the next cell

a4c = ""    # Replace with "linear" or "quadratic"

In [35]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a4c = {a4c}')

a4c = 


## Problem 5: Refit and Evaluate the Final Model

You have used the validation set to make several modeling decisions:

1. whether to use the original or log-transformed `Lot Area`;
2. whether frequency encoding improves on one-hot encoding for `Neighborhood`;
3. whether `Overall Qual` should be modeled with a linear or quadratic relationship.

Based on these experiments, you have now selected a **final model**.

Select the final pipeline based on the modeling decisions made in Problems 2–4. Create a fresh copy of that pipeline and refit it on the complete development set. All other predictors should retain their existing preprocessing.

Then:

1. Fit the complete pipeline on the **development set** (`X_development`, `y_development`), which combines the training and validation data.
2. Use the fitted model to generate predictions for the **untouched test set**.
3. Calculate the test MAE, RMSE, and $R^2$.

> **Important:** The test set should be used **only once**, after all modeling decisions have been made. Do not make any further changes to the model after examining the test results.

### Graded Answers

Assign:

```python id="o1ff4q"
a5a = test MAE
a5b = test RMSE
a5c = test R-squared
```

**Hint:** Use `clone()` from `sklearn.base` to create a fresh, unfitted copy of the selected model pipeline, then fit that copy on `X_development` and `y_development`.

In [36]:
# Your code here, as more cells as needed



In [37]:
# Your answer here, NOT in the next cell

a5a = 0              # Replace with an expression that returns the test mae


In [38]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a5a = ${a5a:,.2f}')

a5a = $0.00


In [39]:
# Your answer here, NOT in the next cell

a5b = 0              # Replace with an expression that returns the test rmse


In [40]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a5b = ${a5b:,.2f}')

a5b = $0.00


In [41]:
# Your answer here, NOT in the next cell

a5c = 0              # Replace with an expression that returns the R2 score


In [42]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a5c = {a5c:.4f}')

a5c = 0.0000


## Problem 6: Multiple Choice

For each question, assign the number of the **best answer** to the corresponding variable. Each question is worth 2 points.

### Part A Graded Answer — Tests: Purpose of the Validation Set

What is the main purpose of the validation set in this homework?

1. To fit the coefficients of each candidate linear regression model.
2. To compare candidate modeling choices without using the final test set.
3. To obtain the final performance estimate after all modeling decisions are complete.
4. To replace missing values before the dataset is split.

Assign your answer to `a6a`.

In [43]:
# Your answer here, NOT in the next cell

a6a = 0             # Replace with the number of the best answer


In [44]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a6a = {a6a}')

a6a = 0


### Part B Graded Answer — Tests: Preprocessing Leakage

Why should imputation, scaling, and frequency encoding be fitted inside a pipeline using the training data?

1. A pipeline guarantees that every model has a lower validation MAE.
2. Linear regression cannot accept data that were prepared before the split.
3. It prevents information from the validation or test data from influencing the fitted preprocessing steps.
4. It makes the fitted regression coefficients equal to correlations.

Assign your answer to `a6b`.

In [45]:
# Your answer here, NOT in the next cell

a6b = 0              # Replace with the number of the best answer


In [46]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a6b = {a6b}')

a6b = 0


### Part C Graded Answer — Tests: Log Transformation

The log transformation made the distribution of `Lot Area` much more symmetric in Homework 03 and also slightly lowered validation MAE in this homework. What is the best conclusion?

1. A more symmetric predictor will always improve a linear regression model.
2. Log transformations should always be used for right-skewed predictors.
3. EDA can suggest a useful transformation, but its predictive value should still be evaluated using validation performance.
4. The transformation should be selected because its histogram looks better, regardless of validation performance.

Assign your answer to `a6c`.

In [47]:
# Your answer here, NOT in the next cell

a6c = 0              # Replace with the number of the best answer


In [48]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a6c = {a6c}')

a6c = 0


### Part D Graded Answer — Tests: Polynomial Regression

Why is a model containing `Overall Qual` and `(Overall Qual)²` still considered a linear regression model?

1. The squared term is ignored when predictions are generated.
2. The prediction remains a linear combination of fitted coefficients, even though one predictor is a nonlinear transformation of an original feature.
3. `PolynomialFeatures` transforms the target so that it becomes linear.
4. A degree-2 model always produces a straight line.

Assign your answer to `a6d`.

In [49]:
# Your answer here, NOT in the next cell

a6d = 0              # Replace with the number of the best answer


In [50]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a6d = {a6d}')

a6d = 0


### Part E Graded Answer — Tests: Final Model Refit

After selecting the final model using validation data, why do we refit its complete pipeline on the combined development set before evaluating it on the test set?

1. To use the test set to select new preprocessing choices.
2. To ensure that the test MAE equals the validation MAE.
3. To eliminate the need for an intercept.
4. To let the selected model learn from all available non-test observations before its one final evaluation.

Assign your answer to `a6e`.

In [51]:
# Your answer here, NOT in the next cell

a6e = 0             # Replace with the number of the best answer


In [52]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print(f'a6e = {a6e}')

a6e = 0


## Problem 7: Discussion

Throughout this homework, you used the validation set to compare alternative models, but you did not use the test set until the final model had been selected.

In one paragraph of approximately 5–7 sentences, explain:

* why it would have been inappropriate to use the test set to choose between the feature-engineering alternatives;
* whether the final test performance was better, worse, or about the same as the validation performance of the selected model;
* why validation and test performance might differ even though each was measured on observations that were not used to fit the model being evaluated;
* what the final test results suggest about how well the modeling process generalized to previously unseen data.

Refer to at least two specific numerical results from your notebook.

In [53]:
# Your answer here, NOT in the next cell

a7 = """
Write your reflection here.
"""

In [54]:
# Graded Answer
# DO NOT change this cell in any way
# Be SURE to run this cell before submitting

print("a7 =")
print(a7)

a7 =

Write your reflection here.

